# SMTNO — GPU training on Google Colab

This notebook is only a launcher for the repository's `.py` scripts. All datasets, results and checkpoints stay in **your private Google Drive**, not GitHub.

Select **Runtime → Change runtime type → GPU** before training. Adjust `DATA_ROOT` to match where you copied your existing results.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import subprocess, sys
import torch
REPO_URL = 'https://github.com/ronitbaishya1/SMTNO.git'
REPO = Path('/content/SMTNO')
if not (REPO/'.git').exists():
    subprocess.run(['git','clone',REPO_URL,str(REPO)],check=True)
else:
    subprocess.run(['git','-C',str(REPO),'pull','--ff-only'],check=True)
SRC = REPO/'src'/'advection_diffusion'
DATA_ROOT = Path('/content/drive/MyDrive/SMTNO/datasets')
OUTPUT_ROOT = Path('/content/drive/MyDrive/SMTNO/GPU_runs')
OUTPUT_ROOT.mkdir(parents=True,exist_ok=True)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('GitHub source:', SRC, 'Device:', DEVICE)


In [ ]:
%pip install -q numpy scipy pandas matplotlib torch


## Step 3C.1 — Advection–diffusion negative-control FNO
Requires the private Step 3B.4 generated dataset in `DATA_ROOT/step3B4`.


In [ ]:
data = DATA_ROOT/'step3B4'
assert data.exists(), f'Copy your Step 3B.4 dataset to {data}'
subprocess.run([sys.executable,str(SRC/'step3c1_train.py'),
  '--indir',str(data),'--outdir',str(OUTPUT_ROOT/'step3C1'),
  '--train-res','32','--test-res','64','--ells','0.01,0.02,0.04',
  '--history','4','--epochs','35','--batch-size','16',
  '--horizons','1,3,5','--eval-stride','4','--device',DEVICE],check=True)


## Step 3C.2A — Repeat FNO training with multiple seeds
Uses the same privately stored Step 3B.4 dataset.


In [ ]:
data = DATA_ROOT/'step3B4'
assert data.exists(), f'Missing {data}'
subprocess.run([sys.executable,str(SRC/'step3c2a_repeats.py'),
  '--indir',str(data),'--outdir',str(OUTPUT_ROOT/'step3C2A'),
  '--seeds','123,456,789','--epochs','70','--horizons','1,3,5,10',
  '--device',DEVICE],check=True)


## Step 3C.2B — Physical-memory transport benchmark
First run `step3c2b_generate.py` and `step3c2b_audit.py` to create a private `step3C2B` dataset (on Mac or Colab). Then train below.


In [ ]:
data = DATA_ROOT/'step3C2B'
assert data.exists(), f'Generate/copy the private MRMT dataset to {data}'
subprocess.run([sys.executable,str(SRC/'step3c2b_train.py'),
  '--indir',str(data),'--outdir',str(OUTPUT_ROOT/'step3C2B_train'),
  '--train-res','32','--test-res','64','--history','4',
  '--epochs','50','--batch-size','16','--horizons','1,5,10',
  '--device',DEVICE],check=True)
